# __Inteligencia Artificial - IUA 2026__

### *Pipeline EDA: Limpieza y preparación de los datos para entrenamiento*

### Integrantes
* **Celeste Antonella Badami** — cbadami845@alumnos.iua.edu.ar
* **Benjamin Vargas** — bvargas161@alumnos.iua.edu.ar

---


## 1. Descripcion del modelo Pipeline (TODO)
Transformaciones:
* 
Eliminaciones:
* Este modelo de pipeline descarta los viajes en los cuales se detecto activa la flag de desvio.


## 2. Preparación del entorno

In [582]:
# importación de librerias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [583]:
# Carga de la muestra de datos preeliminar
df = pd.read_csv('../data/dataset.csv')

### 2.1 Métricas de pipeline

In [584]:
numero_filas_iniciales = len(df)
numero_columnas_iniciales = len(df.columns)

### 2.2 Configuraciones de pipeline

In [585]:
#Añadir calculo de velocidad
calcular_velocidad = True

## 3. Pipeline EDA

### 3.1 Transformación de datos

In [586]:
# Convertir a datetime (utc=True maneja automáticamente el +00)
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)

#### 3.1.2 Implementación de Hora cíclica

In [587]:
segundos_del_dia = (
    df['date_time'].dt.hour * 3600 + 
    df['date_time'].dt.minute * 60 + 
    df['date_time'].dt.second
)

SEGUNDOS_EN_DIA = 24 * 3600  # 86400

# Aplicar Seno y Coseno
df['hora_sin'] = np.sin(2 * np.pi * segundos_del_dia / SEGUNDOS_EN_DIA)
df['hora_cos'] = np.cos(2 * np.pi * segundos_del_dia / SEGUNDOS_EN_DIA)

#### 3.1.3 Label Encoding sobre estados del colectivo

In [588]:
df['current_status_encoded'] = df['current_status'].map({
    'STOPPED_AT': 0, 
    'IN_TRANSIT_TO': 1
})

#### 3.1.3 Codificación de día de la semana

In [589]:
# Extraer día de la semana (0 = Lunes, 6 = Domingo)
df['dia_semana'] = df['date_time'].dt.day_of_week

#### 3.1.3 Ordinal encoding sobre estado de ocupación

In [590]:
mapeo_ocupacion = {
    'MANY_SEATS_AVAILABLE': 1,
    'FEW_SEATS_AVAILABLE': 2,
    'FULL': 3
}

# Realiza el mapeo manteniendo los NaN
df['occupancy_encoded'] = df['occupancy_status'].map(mapeo_ocupacion).fillna(0).astype(int)

> ⚠️ **ADVERTENCIA:** Este bloque realiza el mapeo de categorías pero **deja pasar los valores `NaN` intactos** en `occupancy_encoded` asignandoles un valor 0. Asegúrarse de tratarlos

#### 3.1.4 Mapeo binario para saber si es de día

In [591]:
df['is_day_encoded'] = df['is_day'].astype('Int64')

> ⚠️ **ADVERTENCIA:** Este bloque realiza el mapeo de categorías pero **deja pasar los valores `NaN` intactos** en `is_day_encoded` (los cuales pandas representará como `NaN` de tipo `float64` o `<NA>` de tipo `Int64`). Asegúrarse de tratarlos

### 3.2 Generación de nuevas features

In [592]:
# Crear una variable booleana / binaria para fin de semana
df['es_fin_de_semana'] = df['dia_semana'].isin([5, 6]).astype(int)

### 3.2 Agrupación de datos (Conformación de viajes)

In [593]:
# Crear DataFrame 'viajes' ordenado 
df = df.sort_values(
    by=['bus_id', 'direction_id', 'date_time'],
    ascending=[True, True, True]
).reset_index(drop=True).copy()

# Calcular deltas de tiempo y distancia en variables locales
delta_tiempo_min = (
    df.groupby(['bus_id', 'direction_id'])['date_time']
    .diff()
    .dt.total_seconds() / 60.0
)
delta_distancia = (
    df.groupby(['bus_id', 'direction_id'])['distancia_en_traza']
    .diff()
)

# Detectar puntos de corte de viaje (pausa > 15m, reinicio de traza o cambio de grupo)
es_nuevo_viaje = (
    (delta_tiempo_min > 15) | 
    (delta_distancia < -1.0) | 
    (delta_tiempo_min.isna())
)

# Generar ID único incremental directo (1, 2, 3... N)
df['trip_id'] = es_nuevo_viaje.cumsum()

total_viajes_no_clean = df['trip_id'].nunique()
print(f"Total de viajes identificados: {total_viajes_no_clean}")

Total de viajes identificados: 576


### 3.3 Eliminación de viajes desviados

En este modelo se eliminan todos aquellos viajes que terminaron en desvio

In [594]:
# Obtener la lista/serie de trip_ids que tuvieron al menos un desvío
trips_con_desvio = df[df['es_desvio'] == True]['trip_id'].unique()

# Mantener solo los viajes cuyo trip_id NO esté en la lista de desvíos
df = df[~df['trip_id'].isin(trips_con_desvio)].copy()

total_viajes_clean = df['trip_id'].nunique()
print(f"Total de viajes restantes: {total_viajes_clean} -{(total_viajes_clean-total_viajes_no_clean) / total_viajes_no_clean * 100:.2f}% de viajes eliminados")

Total de viajes restantes: 466 --19.10% de viajes eliminados


### 3.3 Cálculo de velocidad de unidades

In [595]:
if calcular_velocidad:
    # Garantizar orden cronológico estricto dentro de cada viaje
    df = df.sort_values(by=['trip_id', 'date_time']).reset_index(drop=True)

    # Calcular deltas de tiempo (en segundos) y de distancia (en km) agrupando por trip_id
    delta_tiempo_sec = df.groupby('trip_id')['date_time'].diff().dt.total_seconds()
    delta_distancia_km = df.groupby('trip_id')['distancia_en_traza'].diff()

    # Calcular velocidad en km/h: (km / seg) * 3600
    df['velocidad_traza_kmh'] = (delta_distancia_km / delta_tiempo_sec) * 3600.0

    # --- TRATAMIENTO DE BORDES, ANOMALÍAS E IMPUTACIÓN ---

    # a) Frenados o lecturas con delta nulo/negativo (sin avance o duplicado temporal)
    df.loc[(delta_tiempo_sec <= 0) | (delta_distancia_km <= 0), 'velocidad_traza_kmh'] = 0.0

    # b) Teleportaciones o saltos de GPS (> 90 km/h): Asignar directo `current_speed`
    df.loc[df['velocidad_traza_kmh'] > 90.0, 'velocidad_traza_kmh'] = df['current_speed']

    # c) Imputar primeras lecturas del viaje (NaNs por falta de punto anterior) con `current_speed`
    df['velocidad_traza_kmh'] = df['velocidad_traza_kmh'].fillna(df['current_speed'])



#### 3.4 Calculo de distancia restante a paradas

In [596]:
#Importar dataset de distancia de paradas en traza
df_paradas = pd.read_csv('../data/paradas_distancia_traza.csv')

In [597]:
# Asegurar que las claves de cruce coincidan en tipo de dato (string)
df['shape_id'] = df['shape_id'].astype(str)
df_paradas['shape_id'] = df_paradas['shape_id'].astype(str)

#Filtrar df_paradas para seleccionar ÚNICAMENTE stop_sequence y distancia_en_traza_km
df_paradas= df_paradas[['shape_id', 'stop_sequence', 'distancia_parada_traza_km']]

# Hacer el Merge trayendo solo esos datos
df = df.merge(df_paradas, on='shape_id', how='inner')

# Calcular la distancia restante (distancia parada - distancia colectivo)
df['distancia_restante_km'] = df['distancia_parada_traza_km'] - df['distancia_en_traza']


# Filtrar las paradas por las que el colectivo ya pasó (distancia restante <= 0.03 km)
df = df[df['distancia_restante_km'] > 0.03].reset_index(drop=True)

print(f"Tamaño de dataset con distancia a paradas {df.shape[0]} filas y {df.shape[1]} columnas")

Tamaño de dataset con distancia a paradas 133208 filas y 41 columnas


#### 3.5 Calculo de ETA (Target)

In [598]:
# Obtener el timestamp de desaparición (el reporte inmediato siguiente a la última vez que apareció cada parada)
timeline = (
    df[['trip_id', 'date_time']]
    .drop_duplicates()
    .sort_values(by=['trip_id', 'date_time'])
    .reset_index(drop=True)
)
timeline['tiempo_llegada'] = timeline.groupby('trip_id')['date_time'].shift(-1)

ultima_aparicion = (
    df.groupby(['trip_id', 'stop_sequence'])['date_time']
    .max()
    .reset_index()
)


# Unir para obtener el timestamp de llegada de cada parada (el reporte inmediato siguiente a la última vez que apareció cada parada)
llegadas = ultima_aparicion.merge(
    timeline,
    on=['trip_id', 'date_time'],
    how='left'
)[['trip_id', 'stop_sequence', 'tiempo_llegada']]

# Identificar las paradas presentes en el ÚLTIMO registro de cada trip_id (las no alcanzadas)
ultimo_ts_viaje = df.groupby('trip_id')['date_time'].transform('max')
no_alcanzadas = (
    df[df['date_time'] == ultimo_ts_viaje][['trip_id', 'stop_sequence']]
    .drop_duplicates()
)
# Marcar para borrar
no_alcanzadas['descartar'] = True

# Eliminar del dataset todas las filas cuyo stop_sequence figure en ese último registro del viaje
df = df.merge(no_alcanzadas, on=['trip_id', 'stop_sequence'], how='left')
df = df[df['descartar'].isna()].drop(columns=['descartar'])

# Pegar el timestamp de llegada y calcular el ETA en segundos en las muestras restantes
df = df.merge(llegadas, on=['trip_id', 'stop_sequence'], how='left')
df['eta_segundos'] = (df['tiempo_llegada'] - df['date_time']).dt.total_seconds()

# Limpieza final por si queda algún registro nulo o <= 0
df = df[df['eta_segundos'] > 0].reset_index(drop=True)

#### 3.6 Tratamiento de Nulos

In [599]:
resumen_cols = pd.DataFrame({
        'Tipo': df.dtypes,
        'No Nulos': df.notna().sum(),
        'Nulos': df.isna().sum(),
        '% Nulos': (df.isna().mean() * 100).round(2),
        'Únicos': df.nunique()
    })
print(resumen_cols.to_string())

                                          Tipo  No Nulos  Nulos  % Nulos  Únicos
id                                       int64    106831      0     0.00    9011
bearing                                float64     86031  20800    19.47     346
current_status                             str    106831      0     0.00       2
direction_id                             int64    106831      0     0.00       2
occupancy_status                           str    105460   1371     1.28       3
latitude                               float64    106831      0     0.00    6854
longitude                              float64    106831      0     0.00    6952
bus_id                                     str    106831      0     0.00      80
route_id                                 int64    106831      0     0.00       1
route_pattern_id                           str    106831      0     0.00       2
shape_id                                   str    106831      0     0.00       2
agency_id                   

##### 3.6.1 Ocupancy_encoded

In [600]:
# Conteo de registros en 0 en occupancy_encoded
ceros_totales = (df['occupancy_encoded'] == 0).sum()
total_filas = len(df)
porcentaje_ceros = (ceros_totales / total_filas) * 100

print(f"Registros con occupancy_encoded == 0: {ceros_totales:,} de {total_filas:,} ({porcentaje_ceros:.2f}%)")
print("\nDistribución completa de valores:")
print(df['occupancy_encoded'].value_counts().sort_index())

Registros con occupancy_encoded == 0: 1,371 de 106,831 (1.28%)

Distribución completa de valores:
occupancy_encoded
0     1371
1    70516
2    29022
3     5922
Name: count, dtype: int64


In [601]:
# Reemplazar 0 por 1 ya que es la clase mayoritaria, y las mediciones en cero solo representan el 1.28% del total de registros
df['occupancy_encoded'] = df['occupancy_encoded'].replace(0, 1).astype(int)

# Conteo de registros en 0 en occupancy_encoded
ceros_totales = (df['occupancy_encoded'] == 0).sum()
total_filas = len(df)
porcentaje_ceros = (ceros_totales / total_filas) * 100

print(f"Registros con occupancy_encoded == 0 (propagación): {ceros_totales:,} de {total_filas:,} ({porcentaje_ceros:.2f}%)")
print("\nDistribución completa de valores:")
print(df['occupancy_encoded'].value_counts().sort_index())

Registros con occupancy_encoded == 0 (propagación): 0 de 106,831 (0.00%)

Distribución completa de valores:
occupancy_encoded
1    71887
2    29022
3     5922
Name: count, dtype: int64


##### 3.6.2 Meteorología 

In [602]:
# Lista de columnas meteorológicas a imputar (filtramos solo las que existan en df)
cols_clima = [
    'temperature_2m',
    'precipitation',
    'relative_humidity_2m',
    'weather_code',
    'is_day',
    'is_day_encoded'
]
cols_presentes = [c for c in cols_clima if c in df.columns]

# Propagar el valor inmediatamente siguiente (.bfill()) y luego el anterior (.ffill())
df[cols_presentes] = df[cols_presentes].bfill().ffill()

# Verificación de nulos restantes
print("Nulos restantes en columnas de clima:")
print(df[cols_presentes].isna().sum())

Nulos restantes en columnas de clima:
temperature_2m          0
precipitation           0
relative_humidity_2m    0
weather_code            0
is_day                  0
is_day_encoded          0
dtype: int64


#### 3.7 Eliminación de columnas reduntantes

In [603]:
# Varibles innecesarias
df = df.drop(columns=['date_time',
                      'current_status',
                      'occupancy_status',
                      'bus_id',
                      'distancia_fuera_traza',
                      'es_desvio',
                      'longitude',
                      'latitude',
                      'is_day',
                      'distancia_parada_traza_km',
                      'id',
                      'trip_id'])
                      
# Varibles eliminadas por no variar en el dataset
df = df.drop(columns=[
                      'route_id',
                      'route_pattern_id',
                      'shape_id',
                      'agency_id',
                      'line_id',
                      'timezone',
                      'road_closure'])
                      
# Varibles eliminadas que quizas aporten algo si no las eliminamos
# Se consideran eliminar porque varian muy poco
df = df.drop(columns=[
                      'confidence',
                      'bearing',
                      'stop_sequence'
                    ])

### 4. Controles finales

In [604]:
# Dimensiones y peso real en RAM
filas, columnas = df.shape
memoria_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Dimensiones : {filas:,} filas x {columnas} columnas")
print(f"Peso en RAM : {memoria_mb:.2f} MB")

Dimensiones : 106,831 filas x 21 columnas
Peso en RAM : 16.81 MB


In [605]:
if 'trip_id' in df.columns:
    print(f"Viajes únicos (trip_id): {df['trip_id'].nunique():,}")
if 'date_time' in df.columns:
    print(f"Rango temporal : {df['date_time'].min()}  -->  {df['date_time'].max()}")

In [606]:
resumen_cols = pd.DataFrame({
        'Tipo': df.dtypes,
        'No Nulos': df.notna().sum(),
        'Nulos': df.isna().sum(),
        '% Nulos': (df.isna().mean() * 100).round(2),
        'Únicos': df.nunique()
    })
print(resumen_cols.to_string())

                                       Tipo  No Nulos  Nulos  % Nulos  Únicos
direction_id                          int64    106831      0      0.0       2
distancia_en_traza                  float64    106831      0      0.0    8152
temperature_2m                      float64    106831      0      0.0    1863
precipitation                       float64    106831      0      0.0      20
relative_humidity_2m                float64    106831      0      0.0      55
weather_code                        float64    106831      0      0.0      10
current_speed                         int64    106831      0      0.0      36
free_flow_speed                       int64    106831      0      0.0      31
current_travel_time                   int64    106831      0      0.0     174
freeflow_travel_time                  int64    106831      0      0.0      76
hora_sin                            float64    106831      0      0.0    8564
hora_cos                            float64    106831      0    

In [607]:
print("\n--- ESTADÍSTICAS DE eta_segundos ---")
print(df['eta_segundos'].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2).to_string())


--- ESTADÍSTICAS DE eta_segundos ---
count    106831.00
mean       1008.25
std         738.06
min          36.26
1%           75.10
25%         415.70
50%         849.72
75%        1447.36
99%        3157.57
max        4835.14


### 5. Exportación de dataset

In [608]:
df.head(500).to_csv('../data/eda_dataset.csv', index=False, encoding='utf-8-sig')